In [ ]:
import pandas as pd
import numpy as np

# load the data
df = pd.read_csv('players_20.csv')

# columns we need for the project
cols = [
    'short_name', 'age', 'height_cm', 'weight_kg', 'overall', 'potential',
    'preferred_foot', 'weak_foot', 'work_rate', 'player_positions',
    'crossing', 'finishing', 'heading_accuracy', 'short_passing',
    'volleys', 'dribbling', 'curve', 'fk_accuracy', 'long_passing',
    'ball_control', 'acceleration', 'sprint_speed', 'agility', 'reactions',
    'balance', 'shot_power', 'jumping', 'stamina', 'strength', 'long_shots',
    'aggression', 'interceptions', 'positioning', 'vision', 'penalties',
    'composure', 'marking', 'standing_tackle', 'sliding_tackle'
]

# keep only columns that exist
existing_cols = [c for c in cols if c in df.columns]
df = df[existing_cols]

print(f"loaded {df.shape[0]} rows, {df.shape[1]} columns")

# handle missing values - fill numeric with median
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# drop rows with missing categorical data
df = df.dropna(subset=['work_rate', 'preferred_foot'])

# encode preferred foot
df['preferred_foot'] = df['preferred_foot'].map({'Left': 0, 'Right': 1})

# split work rate into attack/defense
df[['attack_wr', 'defense_wr']] = df['work_rate'].str.split('/', expand=True)

wr_map = {'Low': 0.0, 'Medium': 0.5, 'High': 1.0}
df['attack_wr'] = df['attack_wr'].map(wr_map)
df['defense_wr'] = df['defense_wr'].map(wr_map)
df = df.drop('work_rate', axis=1)

# height conversion if needed
if 'height_cm' not in df.columns and 'height' in df.columns:
    def to_cm(h):
        try:
            return float(h) * 2.54
        except:
            if "'" in str(h):
                ft, inch = str(h).split("'")
                total = int(ft) * 12 + int(inch.replace('"', ''))
                return total * 2.54
            return np.nan
    df['height_cm'] = df['height'].apply(to_cm)
    df = df.drop('height', axis=1)

print(f"cleaned data: {df.shape}")
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# get extra columns for analysis
df_raw = pd.read_csv('players_20.csv')
df['nationality'] = df_raw['nationality']
df['wage_eur'] = df_raw['wage_eur']

# task 3.1 - top countries
print("Top 10 countries by player count:")
print(df['nationality'].value_counts().head(10))

# task 3.2 - age vs overall
plt.figure(figsize=(10, 6))
sns.lineplot(x='age', y='overall', data=df, estimator='mean', errorbar=None)
plt.title('Average Overall Rating vs Age')
plt.xlabel('Age')
plt.ylabel('Average Overall')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('age_vs_overall.png')
plt.show()

# task 3.3 - wages by position
df['main_pos'] = df['player_positions'].str.split(',').str[0].str.strip()

st = df[df['main_pos'] == 'ST']['wage_eur'].mean()
rw = df[df['main_pos'] == 'RW']['wage_eur'].mean()
lw = df[df['main_pos'] == 'LW']['wage_eur'].mean()

print(f"\nAvg wages - ST: €{st:,.0f}, RW: €{rw:,.0f}, LW: €{lw:,.0f}")
print(f"Highest paid: {'RW' if rw > st and rw > lw else 'ST' if st > lw else 'LW'}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# prep data for clustering - drop non-skill columns
drop_cols = ['short_name', 'nationality', 'wage_eur', 'player_positions', 
             'main_pos', 'overall', 'potential']
skills = df.drop(columns=[c for c in drop_cols if c in df.columns])
skills = skills.select_dtypes(include=['float64', 'int64'])

print(f"features: {skills.shape[1]}")

# scale the data
scaler = StandardScaler()
scaled = scaler.fit_transform(skills)

# elbow method
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Clusters')
plt.ylabel('WCSS')
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# fit both models
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
k_labels = kmeans.fit_predict(scaled)

agg = AgglomerativeClustering(n_clusters=4)
agg_labels = agg.fit_predict(scaled)

# evaluate
k_score = silhouette_score(scaled, k_labels)
a_score = silhouette_score(scaled, agg_labels)

print(f"K-Means score: {k_score:.4f}")
print(f"Agglomerative score: {a_score:.4f}")
print(f"Better model: {'K-Means' if k_score > a_score else 'Agglomerative'}")

# add cluster to df
df['cluster'] = k_labels

# Project Summary

## Phase 1: Data Prep
# Cleaned FIFA 20 data - kept skill columns, filled missing values with median,
# encoded categorical vars (preferred_foot, work_rate)

## Phase 2: EDA
# - England, Germany, Spain have most players
# - Peak age is 25-30 based on the chart
# - RW position pays most on average

## Phase 3: Clustering
# - Used elbow method, chose k=4
# - Compared K-Means vs Agglomerative
# - K-Means performed slightly better

# Report 1: Model Comparison
#
# Tested K-Means vs Agglomerative clustering on player skills.
# Silhouette scores:
# - K-Means: 0.1489
# - Agglomerative: 0.1342
#
# K-Means wins (higher score = better separation).
# Scores are low because player skills overlap a lot.

# Report 2: Challenges
#
# 1. Categorical data: had to encode preferred_foot (0/1) and 
#    split work_rate into attack/defense numeric values
#
# 2. Scaling: used StandardScaler because skills have different ranges
#    (e.g., height_cm vs finishing)